---

In [1]:
import os
import datetime
import pandas as pd

## Открытие файлов:

### Остатки:

In [2]:
# Имена файлов
stock_file_names = os.listdir(f'{os.curdir}/stock')

In [3]:
# Пути к файлам (Путь к директории + Имя файла)
stock_file_paths = [f'{os.curdir}/stock/' + x for x in stock_file_names if x.endswith('.csv')]

In [4]:
stock_file_paths

['./stock/stock_2025_04_30.csv']

Проверка количества файлов с остатками:

In [5]:
if len(stock_file_paths) > 1:
    raise Exception('Обнаружено более 1 файла с остатками')
else:
    stock_df = pd.read_csv(stock_file_paths[0], sep=';')

In [6]:
stock_df['trans_date'] = stock_df['trans_date'].astype('datetime64[ns]')

Проверка дубликатов в остатках:

In [7]:
# Количество дубликатов по ключевым полям
stock_duplicates = stock_df.duplicated(subset=['item_id', 'location_id']).sum()

if stock_duplicates != 0:
    raise Exception('Необходима корректировка остатков. Найдены дубликаты по ключевым полям item_id и location_id')

### Движения

In [8]:
# Имена файлов
movements_file_names = os.listdir(f'{os.curdir}/invent_trans')

In [9]:
# Пути к файлам (Путь к директории + Имя файла)
movements_file_paths = [f'{os.curdir}/invent_trans/' + x for x in movements_file_names if x.endswith('.csv')]

In [10]:
movements_file_paths

['./invent_trans/invent_trans_2025_05.csv',
 './invent_trans/invent_trans_2025_06.csv',
 './invent_trans/invent_trans_2025_07.csv']

---

In [11]:
# Переменная, ограничивающая количество файлов для конкатенации
file_max = 5

# Инициализация Датафрейма
movements_df = pd.DataFrame()

# Конкатенация файлов 
if len(movements_file_paths) < file_max:
    movements_df = pd.concat(
        [pd.read_csv(x, sep=';') for x in movements_file_paths]
    )

# Проверка конкатенации
if len(movements_df) == 0:
    raise Exception('Ошибка конкатенации') 

In [12]:
movements_df['trans_date'] = movements_df['trans_date'].astype('datetime64[ns]')

Дубликаты строк:

In [13]:
# Количество всех строк, которые имеют повторы
int(movements_df.duplicated(keep=False).sum())

405102

In [14]:
# Количество повторов 
int(movements_df.duplicated(keep='first').sum())

224499

In [15]:
# Первые десять
movements_df[movements_df.duplicated(keep=False)].sort_values(
    by=list(movements_df.columns)
).head(10)

,item_id,location_id,trans_date,qty,cost_amount
309331,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-10,-1.0,-117.8
310922,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-10,-1.0,-117.8
100602,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-23,6.0,706.8
100614,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-23,6.0,706.8
20578,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-31,-1.0,-117.8
21715,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-05-31,-1.0,-117.8
79522,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-06-20,10.0,1178.0
79540,0000CD49FA7DFE1158A3BB3DBE8E1489,FC081C55582CED11A62CD892C0003F2A,2025-06-20,10.0,1178.0
162256,0000EB60690F9E117260106505009C4A,FC081C55582CED11A62CD892C0003F2A,2025-07-28,-1.0,-311.1
163532,0000EB60690F9E117260106505009C4A,FC081C55582CED11A62CD892C0003F2A,2025-07-28,-1.0,-311.1


Распределение дубликатов:

In [16]:
movements_df['month'] = movements_df['trans_date'].dt.month

In [17]:
# Распределение повторов по месяцам
movements_df[movements_df.duplicated(keep='first')]\
    .groupby('month').trans_date.count()

month
5    70818
6    73673
7    80008
Name: trans_date, dtype: int64

In [18]:
movements_df.drop(['month'],axis=1, inplace=True)

---

### Календарь наличия

In [19]:
# Справочник вида локация-товар
item_locateion_dim = pd.concat([movements_df[['location_id','item_id']], stock_df[['location_id','item_id']]]).drop_duplicates()

In [20]:
# Объявлеие начала и конца необходимого периода
start_day = '2025-04-30'
end_day = '2025-07-31'

# Генерация дней
date_index = pd.date_range(start=start_day, end=end_day, freq='D')

date_dim = pd.DataFrame(date_index, columns=['stock_date'])

**Матрица вида:** `Календарный день`-`Локация`-`Товар`

In [21]:
item_location_matrix = date_dim.merge(
    item_locateion_dim,
    how='cross'
)

## Агрегация данных

Расчёт дневных итогов для каждого товара с учётом локации:

In [22]:
movements_df_total = movements_df.groupby(
    ['location_id','item_id','trans_date']
).agg(
    {'qty':'sum',
     'cost_amount':'sum'}
).reset_index()

**Объединение данных остатков и транзакций:**

In [23]:
df_history = pd.concat([stock_df, movements_df_total]).sort_values(by=['location_id','item_id','trans_date'])

**Матрица остатков:**

In [24]:
df_matrix = item_location_matrix.merge(
    df_history,
    how='left',
    left_on=['item_id','location_id','stock_date'],
    right_on=['item_id','location_id','trans_date'])

Сортировка массива данных для корректного расчёта остатков:

In [25]:
df_matrix.sort_values(
    by=['location_id','item_id','stock_date'], inplace=True
)

In [26]:
# Заполнение пропусков в столбцах факта
df_matrix.fillna(
    {'qty':0, 
     'cost_amount':0,
     'trans_date':df_matrix['stock_date']}, 
    inplace=True
)

**Расчёт остатков по дням** (кумулятивная сумма от начальной даты):

Остаток по `qty`:

In [27]:
df_matrix['stock_qty'] = df_matrix.groupby(
    ['location_id','item_id']
)['qty'].cumsum()

Остаток по `cost_amount`:

In [28]:
df_matrix['stock_cost_amount'] = df_matrix.groupby(
    ['location_id','item_id']
)['cost_amount'].cumsum()

# Сохранение результатов подсчёта:

Сортировка значений даты:

In [29]:
date_values = sorted(date_dim.stock_date.dt.date.unique())

Перебор факта остатков по дням:

In [30]:
for d_value in date_values:
    
    # Фильтр по дню остатка
    filter_mask = ( df_matrix['trans_date'] == d_value.strftime('%Y-%m-%d') )

    # Преобразование для сохранения
    # Фильтрация
    day_stock_df = df_matrix[filter_mask][ 
        # Выбираем нужные столбы
        ['item_id','location_id','trans_date','stock_qty','stock_cost_amount']
            ].rename(
                # Переименование столбцов
                columns={'stock_qty':'qty','stock_cost_amount':'cost_amount'}
                    )
    # Форматирование даты
    d_str_value = d_value.strftime('%Y_%m_%d')

    # Сохранение в CSV-файл
    day_stock_df.to_csv(f'{os.curdir}/stock/counted_stock/stock_{d_str_value}.csv', index=False, sep=';')

---